In [ ]:
import base64
import io
import json
import os
import re
import stringcase
import voxeloo
from dataclasses import dataclass, replace
from glob import glob
from string import Template
from typing import Dict
from PIL import Image

In [ ]:
def load_texture(path):
    return Image.open(f"../../data/{path}")

In [ ]:
def flip_texture(img):
    return img.transpose(Image.FLIP_TOP_BOTTOM)

In [ ]:
def encode_texture(image):
    with io.BytesIO() as output:
        image.save(output, format="PNG")
        return base64.b64encode(output.getvalue()).decode("utf-8")

In [ ]:
@dataclass
class Sample:
    side: str
    top: str
    bottom: str
    mrea: str
    mreaside: str
    mreatop: str
    mreabottom: str

@dataclass
class Block:
    black: Dict[str, Sample]
    white: Dict[str, Sample]

In [ ]:
CUBE_TEXTURE_TEMPLATE = Template("""{
  "x_neg": "$side",
  "x_pos": "$side",
  "y_neg": "$bottom",
  "y_pos": "$top",
  "z_neg": "$side",
  "z_pos": "$side"
}""")

SAMPLE_TEMPLATE = Template("""{
  "color": $color,
  "mrea": $mrea
}"""
)

BLOCK_TEMPLATE = Template("""{
  "name": "$name",
  "label": "$label",
  "index": 1,
  "white": [
    $white
  ],
  "black": [
    $black
  ]
}"""
)

def json_format(s: str):
    return json.dumps(json.loads(s), indent=2)

def get_mrea(sample, side):
  potentialTextures = ['mrea'+side, 'mrea']
  for attr in potentialTextures:
    if hasattr(sample, attr):
      potential = getattr(sample, attr)
      if potential != "":
        return potential
  return "blocks/textures/shared/default_mrea.png"

def block_to_json(name: str, block: Block):
    black = [
        SAMPLE_TEMPLATE.substitute({
            "color": CUBE_TEXTURE_TEMPLATE.substitute({
              "side": encode_texture(flip_texture(load_texture(sample.side))),
              "top": encode_texture(flip_texture(load_texture(sample.top))),
              "bottom": encode_texture(flip_texture(load_texture(sample.bottom))),
            }),
            "mrea": CUBE_TEXTURE_TEMPLATE.substitute({
              "side": encode_texture(flip_texture(load_texture(get_mrea(sample, "side")))),
              "bottom": encode_texture(flip_texture(load_texture(get_mrea(sample, "bottom")))),
              "top": encode_texture(flip_texture(load_texture(get_mrea(sample, "top"))))
            })
        })
        for sample in block.black.values()
    ]
    white = [
        SAMPLE_TEMPLATE.substitute({
            "color": CUBE_TEXTURE_TEMPLATE.substitute({
              "side": encode_texture(flip_texture(load_texture(sample.side))),
              "top": encode_texture(flip_texture(load_texture(sample.top))),
              "bottom": encode_texture(flip_texture(load_texture(sample.bottom))),
            }),
            "mrea": CUBE_TEXTURE_TEMPLATE.substitute({
              "side": encode_texture(flip_texture(load_texture(get_mrea(sample, "side")))),
              "bottom": encode_texture(flip_texture(load_texture(get_mrea(sample, "bottom")))),
              "top": encode_texture(flip_texture(load_texture(get_mrea(sample, "top"))))
            })
        })
        for sample in block.white.values()
    ]
    
    return json_format(
        BLOCK_TEMPLATE.substitute({
            "name": stringcase.titlecase(name),
            "label": "1",
            "black": ",".join(black),
            "white": ",".join(white),
        })
    )

In [ ]:
@dataclass
class Rename:
    old: str
    new: str

def rename(blocks, old_name, new_name):
    ret = []
    
    def update_sample(sample):
        n = len(old_name)
        ret.append(Rename(sample.side, sample.side.replace(old_name, new_name)))
        ret.append(Rename(sample.top, sample.top.replace(old_name, new_name)))
        ret.append(Rename(sample.bottom, sample.bottom.replace(old_name, new_name)))
        if len(sample.mrea) > 0:
            ret.append(Rename(sample.mrea, sample.mrea.replace(old_name, new_name)))
        if len(sample.mreaside) > 0:
            ret.append(Rename(sample.Rename, sample.Rename.replace(old_name, new_name)))
        if len(sample.mreatop) > 0:
            ret.append(Rename(sample.mreatop, sample.mreatop.replace(old_name, new_name)))
        if len(sample.mreabottom) > 0:
            ret.append(Rename(sample.mreabottom, sample.mreabottom.replace(old_name, new_name))) 
    
    for key, block in blocks.items():
        if key.startswith(old_name):
            block = replace(block)
            for sample in block.black.values():
                update_sample(sample)
            for sample in block.white.values():
                update_sample(sample)
    return ret

In [ ]:
blocks = {}
for path in glob("../../data/blocks/textures/*.png"):
    filename = os.path.basename(path)
    name, tile, variant, side = re.match("(.+)_([^_]+)_([^_]+)_([^_]+).png", filename).groups()
    
    block = blocks.setdefault(name, Block({}, {}))
    sample = getattr(block, tile).setdefault(variant, Sample("", "", "", "", "", "", ""))
    setattr(sample, side, f"blocks/textures/{filename}")

In [ ]:
renames = []

In [ ]:
rocks = [
    #"basalt",
    #"clay",
    "cobblestone",
    #"granite",
    "limestone",
    "quartzite",
    "stone",
]

for rock in rocks:
    #renames.extend(rename(blocks, f"{rock}_brick2", f"{rock}_pavers"))
    #renames.extend(rename(blocks, f"{rock}_chiselled", f"{rock}_carved"))
    #renames.extend(rename(blocks, f"{rock}_smooth", f"{rock}_polished"))
    renames.extend(rename(blocks, f"{rock}_pavers", f"{rock}_brick"))

In [ ]:
renames.extend(rename(blocks, f"quartzsite_brick", f"quartzite_brick"))
renames.extend(rename(blocks, f"quartzsite_carved", f"quartzite_carved"))
renames.extend(rename(blocks, f"quartzsite_polished", f"quartzite_polished"))
renames.extend(rename(blocks, f"quartzsite_pavers", f"quartzite_pavers"))
renames.extend(rename(blocks, f"quartzsite", f"quartzite"))

In [ ]:
woods = [
    "birch",
    "oak",
    "rubber",
]

for wood in woods:
    renames.extend(rename(blocks, f"{wood}_lumber2", f"{wood}_lumber"))

In [ ]:
crates = [
    "wood",
    "silver",
    "gold",
    "diamond",
]

for crate in crates:
    renames.extend(rename(blocks, f"{crate}_chest", f"{crate}_crate"))

In [ ]:
renames.extend(rename(blocks, f"roof_tiles", f"clay_shingles"))

In [ ]:
renames.extend(rename(blocks, f"dirt3", f"dirt"))

In [ ]:
renames

In [ ]:
for rename in renames:
    os.rename(f"../../data/{rename.old}", f"../../data/{rename.new}")